# Kapitel 6 – Stufe 1: Explorative Datenanalyse (EDA)
**Masterarbeit | Kapitel 6.1**  
Autor: Ayumi Nojima | April 2026

---
Dieses Notebook deckt die explorative Datenanalyse ab:
- **6.1.1** Deskriptive Statistiken des Exposure-Index
- **6.1.2** Verteilungsanalyse E_j, E^sub_j, E^aug_j nach Hauptgruppe
- **6.1.3** Korrelationsanalyse zwischen Indexkomponenten und ΔBFS_j
- **6.1.4** Clusteranalyse (K-Means auf standardisierten Fähigkeitsvektoren)
- **6.1.5** PCA-Visualisierung


## 0. Konfiguration und Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

# ── Pfade ──────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if (_cwd / "..").resolve().joinpath("data").exists() else _cwd
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = Path.cwd()

PROCESSED = REPO_ROOT / "data" / "processed"
ANALYSIS  = PROCESSED / "analysis_prep"
OUTPUT    = REPO_ROOT / "data" / "output"
FIGURES   = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# ── Parameter ──────────────────────────────────────────────────────────────
BFS_BASE_YEAR   = 2022
BFS_TARGET_YEAR = 2024

# ── Daten laden ────────────────────────────────────────────────────────────
final         = pd.read_csv(ANALYSIS / "final_sample.csv")
skill_vectors = pd.read_csv(PROCESSED / "skill_vectors_standardized.csv", index_col="soc_code")

print(f"Finales Sample geladen:   {len(final)} Berufe")
print(f"Skill-Vektoren geladen:   {skill_vectors.shape[0]} Berufe × {skill_vectors.shape[1]} Dimensionen")
print(f"Hauptgruppen: {final['main_group'].value_counts().sort_index().to_dict()}")
print("Konfiguration geladen ✓")


FileNotFoundError: [Errno 2] No such file or directory: '/workspaces/Master_Thesis/data/processed/analysis_prep/final_sample.csv'

---
## 6.1.1 Deskriptive Statistiken

Übersicht über die zentralen Tendenz- und Streuungsmasse der drei Indexkomponenten  
sowie der abhängigen Variable ΔBFS_j, differenziert nach CH-ISCO-Hauptgruppe.


In [ ]:
# ── Gesamtdeskriptive ─────────────────────────────────────────────────────
desc_cols = ["E_j", "E_sub_j", "E_aug_j", "delta_bfs"]
print("=== Deskriptive Statistiken – Gesamtsample ===")
print(final[desc_cols].describe().round(3).to_string())

print()
print("=== Deskriptive Statistiken – nach Hauptgruppe ===")
for hg in [1, 2, 3]:
    subset = final[final["main_group"] == hg]
    label = {1: "HG 1 – Führungskräfte", 2: "HG 2 – Akademisch", 3: "HG 3 – Techniker"}[hg]
    print(f"\n{label} (n={len(subset)}):")
    print(subset[desc_cols].describe().round(3).loc[["mean","std","min","max"]].to_string())


In [ ]:
# ── Anteil substituierbarer vs. augmentierbarer Berufe ────────────────────
final["exposure_category"] = pd.cut(
    final["E_j"],
    bins=[0, 0.33, 0.66, 1.0],
    labels=["Niedrig (0–0.33)", "Mittel (0.33–0.66)", "Hoch (0.66–1.0)"],
    include_lowest=True
)
cat_dist = final.groupby(["main_group", "exposure_category"], observed=True).size().unstack(fill_value=0)
print("=== Exposure-Kategorien nach Hauptgruppe ===")
print(cat_dist.to_string())
print()
pct = cat_dist.div(cat_dist.sum(axis=1), axis=0).round(3) * 100
print("=== In Prozent ===")
print(pct.to_string())


---
## 6.1.2 Verteilungsanalyse


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Verteilung der Exposure-Indizes nach CH-ISCO-Hauptgruppe", fontsize=14, fontweight="bold")

hg_labels = {1: "HG 1\nFührungskräfte", 2: "HG 2\nAkademisch", 3: "HG 3\nTechniker"}
colors     = {1: "#2196F3", 2: "#4CAF50", 3: "#FF9800"}

for col_idx, (metric, title) in enumerate([
    ("E_j",     "LLM-Exposure-Index E_j"),
    ("E_sub_j", "Substitutions-Subindex E^sub"),
    ("E_aug_j", "Augmentations-Subindex E^aug"),
]):
    # Boxplot
    ax = axes[0, col_idx]
    data_by_hg = [final[final["main_group"] == hg][metric].dropna() for hg in [1, 2, 3]]
    bp = ax.boxplot(data_by_hg, labels=[hg_labels[h] for h in [1,2,3]],
                    patch_artist=True, medianprops={"color": "black", "linewidth": 2})
    for patch, hg in zip(bp["boxes"], [1,2,3]):
        patch.set_facecolor(colors[hg])
        patch.set_alpha(0.7)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel("Index-Wert [0,1]")
    ax.set_ylim(0, 1)

    # KDE
    ax = axes[1, col_idx]
    for hg in [1, 2, 3]:
        data = final[final["main_group"] == hg][metric].dropna()
        data.plot.kde(ax=ax, label=hg_labels[hg], color=colors[hg], linewidth=2)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Index-Wert [0,1]")
    ax.set_ylabel("Dichte")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / "6_1_exposure_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Abbildung gespeichert → output/figures/6_1_exposure_distribution.png ✓")


---
## 6.1.3 Korrelationsanalyse


In [ ]:
# ── Korrelationsmatrix ────────────────────────────────────────────────────
corr_cols = ["E_j", "E_sub_j", "E_aug_j", "delta_bfs", "s_ij"]
corr_matrix = final[corr_cols].corr(method="pearson")

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    xticklabels=corr_cols, yticklabels=corr_cols
)
ax.set_title("Pearson-Korrelationsmatrix – Indexkomponenten und ΔBFS_j", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / "6_1_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print(corr_matrix.round(3).to_string())


In [ ]:
# ── Scatterplot E_j vs. ΔBFS_j nach Hauptgruppe ──────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

for hg in [1, 2, 3]:
    subset = final[final["main_group"] == hg].dropna(subset=["E_j", "delta_bfs"])
    ax.scatter(subset["E_j"], subset["delta_bfs"],
               label=hg_labels[hg], color=colors[hg], alpha=0.65, s=60, edgecolors="white", linewidth=0.5)

# Regressionslinie (Gesamt)
valid = final.dropna(subset=["E_j", "delta_bfs"])
z = np.polyfit(valid["E_j"], valid["delta_bfs"], 1)
p = np.poly1d(z)
x_line = np.linspace(0, 1, 100)
ax.plot(x_line, p(x_line), "k--", linewidth=1.5, alpha=0.6, label="Regressionslinie (gesamt)")

ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
ax.set_xlabel("LLM-Exposure-Index E_j [0,1]", fontsize=12)
ax.set_ylabel("Beschäftigungsveränderung ΔBFS_j (%)", fontsize=12)
ax.set_title("E_j vs. ΔBFS_j nach CH-ISCO-Hauptgruppe", fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES / "6_1_scatter_Ej_deltabfs.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 6.1.4 Clusteranalyse (K-Means)

Ergänzende Clusteranalyse auf standardisierten Fähigkeitsvektoren zur Identifikation  
latenter Berufsgruppen-Muster jenseits der ISCO-Klassifikation.  
Optimale Clusteranzahl via Elbow-Methode und Silhouette-Score.


In [ ]:
# ── Skill-Vektoren auf finales Sample einschränken ────────────────────────
skill_sub = skill_vectors.loc[skill_vectors.index.isin(final["soc_code"])].fillna(0)
final_with_skills = final.set_index("soc_code").join(skill_sub, how="inner")

X = skill_sub.values
print(f"Clusteranalyse-Input: {X.shape[0]} Berufe × {X.shape[1]} standardisierte Dimensionen")

# ── Elbow + Silhouette ─────────────────────────────────────────────────────
inertias   = []
silhouettes = []
K_range    = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(K_range, inertias, "o-", color="#2196F3", linewidth=2)
ax1.set_xlabel("Anzahl Cluster k")
ax1.set_ylabel("Inertia (Within-Cluster-SS)")
ax1.set_title("Elbow-Methode")

ax2.plot(K_range, silhouettes, "o-", color="#4CAF50", linewidth=2)
ax2.set_xlabel("Anzahl Cluster k")
ax2.set_ylabel("Silhouette-Score")
ax2.set_title("Silhouette-Score")
ax2.axhline(max(silhouettes), color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(FIGURES / "6_1_cluster_elbow.png", dpi=150, bbox_inches="tight")
plt.show()

best_k = K_range[silhouettes.index(max(silhouettes))]
print(f"Optimale Clusteranzahl (max. Silhouette): k={best_k} (Score={max(silhouettes):.3f})")


In [ ]:
# ── Finale K-Means Lösung ─────────────────────────────────────────────────
K_FINAL = best_k  # ggf. manuell überschreiben: K_FINAL = 4

km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X)

# Cluster-Zuordnung ins finale Sample eintragen
skill_sub_reset = skill_sub.reset_index()
cluster_df = pd.DataFrame({"soc_code": skill_sub.index, "cluster": cluster_labels})
final = final.merge(cluster_df, on="soc_code", how="left")

# Cluster-Profil: mittlerer E_j und Hauptgruppen-Verteilung
cluster_profile = final.groupby("cluster").agg(
    n_berufe=("soc_code", "count"),
    E_j_mean=("E_j", "mean"),
    E_sub_mean=("E_sub_j", "mean"),
    E_aug_mean=("E_aug_j", "mean"),
    delta_bfs_mean=("delta_bfs", "mean"),
    hg_mode=("main_group", lambda x: x.mode()[0])
).round(3)

print(f"=== Cluster-Profile (k={K_FINAL}) ===")
print(cluster_profile.to_string())


## 6.1.5 PCA-Visualisierung

In [ ]:
# ── PCA auf 2 Komponenten für Visualisierung ──────────────────────────────
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("PCA der standardisierten Fähigkeitsvektoren", fontsize=13, fontweight="bold")

# Links: nach Cluster
scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1],
                        c=cluster_labels, cmap="tab10", alpha=0.7, s=50, edgecolors="white", linewidth=0.3)
ax1.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% Varianz)")
ax1.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% Varianz)")
ax1.set_title(f"Cluster-Zugehörigkeit (k={K_FINAL})")
plt.colorbar(scatter1, ax=ax1, label="Cluster")

# Rechts: nach Hauptgruppe
for hg in [1, 2, 3]:
    mask = final.set_index("soc_code").reindex(skill_sub.index)["main_group"] == hg
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 1],
                label=hg_labels[hg], color=colors[hg], alpha=0.7, s=50, edgecolors="white", linewidth=0.3)
ax2.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% Varianz)")
ax2.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% Varianz)")
ax2.set_title("CH-ISCO-Hauptgruppe")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / "6_1_pca_visualization.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Erklärte Varianz PC1+PC2: {sum(pca.explained_variance_ratio_[:2])*100:.1f}%")


In [ ]:
# ── EDA-Datensatz mit Cluster-Labels speichern ────────────────────────────
final.to_csv(ANALYSIS / "final_sample_clustered.csv", index=False)
print("Gespeichert → data/processed/analysis_prep/final_sample_clustered.csv ✓")
print()
print("=== EDA Zusammenfassung ===")
print(f"  Berufe im Sample:      {len(final)}")
print(f"  Berufe mit ΔBFS_j:     {final['delta_bfs'].notna().sum()}")
print(f"  Ausreisser (IQR):      {final['is_outlier'].sum() if 'is_outlier' in final.columns else 'N/A'}")
print(f"  Cluster (k={K_FINAL}):         {K_FINAL}")
print(f"  Mittlerer E_j:         {final['E_j'].mean():.3f}")
